![Wolfe](https://s3.amazonaws.com/lquant-images/wolfe_luo.jpg)

# Table of Contents<a name="TOC"></a> 

1. [Introduction](#intro)
    1. [Prerequisites](#prereq)
    1. [Scenario](#scenario)
    1. [Authentication and Connection](#auth_conn)
1. [The Catalog](#catalog)
    1. [Universes](#univ)
    1. [Factors](#factor)
    1. [Portfolios](#port)
    1. [Templates](#temp)
1. [Building a Custom Risk Model](#riskmodel)
    1. [Risk Model Builder Class](#rmb)
    1. [Getting Data from Request](#req_data)
        1. [Factor Covariance Matric](#.cov)
        1. [Security Exposures](#.exp)
        1. [Factor Mimicking Portfolio](#.fmp)
        1. [Security Master](#.idm)
        1. [Factor Info file](#.info)
        1. [Security Meta](#.meta)
        1. [Security Specific Risk](#.rsk)
        1. [Factor Returns (over the period)](#rtn)
1. [Job Requests](#job_reqs)
    1. [Job Request Content](#req_cont)
    1. [Job Info](#job_info)
    1. [Job Logs](#job_logs)
    1. [Searching for a Job](#find_job)
1. [Portfolio Attribution](#attr1)
    1. [Attribution Results](#attr1res)
        1. [Correlations](#corr1)
        1. [Factor Exposures](#fact_exp1)
1. [Portfolio Optimization](#opt1)
    1. [Optimizer Class](#opt_class)
        1. [Objective](#objec)
        1. [Constraints](#const)
    1. [Inspecting the Result](#inspect_res1)
        1. [Optimized Weights](#opt_weight1)
        1. [Ex-ante Alpha](#exalpha)
        1. [Ex-ante Risk](#exrisk)
        1. [Portfolio Turnover](#port_turn)
    1. [Creating the Optimized Portfolio (Could be doing wrong)](#opt_port1)

##### At the end of each chapter, there is a link to return to the table of contents.

[To Do List (delete when done)](#todo)

# Luo's QES - Python Microservices API <a name="intro"></a>

## <u> Prerequisites<a name="prereq"></a>
Before starting, make sure to follow the steps below:
1. Copy the [pyqes]( https://github.com/wolferesearch/docs/tree/master/micro-services/api/python/pyqes) python library from Github to your local working directory. 
2. Ensure you have the [pandas](https://pandas.pydata.org/) and [requests](https://pypi.org/project/requests/) packages in your python kernel. 

In [29]:
# Remember, make sure the 'pyques' folder is in your working directory
from pyqes import micsvc

import pandas as pd
import requests

## <u>Scenario<a name="scenario"></a>
You have a net long portfolio that has seen significant tailwinds due to systematic factors.

You want to build an on-the-fly risk model with some customization of the QES standard US risk model template, and use that to:
1) To build a tradable hedge basket for your portfolio of US stocks. Or alternatively, 
2) Optimize that portfolio sourcing idio risk from stocks already in the portfolio, while minimizing systematic risk.

This guide will take you through the use of the Microservices (micsvc) API to achieve the above, using the following progression:
1. Crafting and fine tuning a <u>**custom risk model**</u> for a portfolio (with and without the 1 month reversion factor), and change the estimation universe.
2. See an initial <u>**attribution**</u> for the portfolio and seeing the results for the 2 risk model variations. 
3. <u>**Optimizing**</u> the hedge or portfolio using either variation of the risk models (standard US custom) and set risk objectives and tradability constraints.
    * After optimization, you will also see portfolio and hedge attributions and the optimized portfolio attribution.
4. See the <u>**performance metrics**</u> of initial, hedge, and optimized portfolios including side-by-side comparisons.

## <u>Authentication and Connection<a name="auth_conn"></a>

The API is protected using Username and Password. In case you have not received it, please [email](mailto:luo.qes@wolferesearch.com) to apply for API account. 

The connection object is the gateway to accessing the API. It allows you to access the catalog, portfolios, templates, risk models, etc. 

<span style="color:red">*FOR ME: Replace parameters with ***** when done</span>

In [58]:
# create a connection object to interact with the server
user_name = 'hjain'
connection = micsvc.Connection(username = user_name, password = 'TestUser123')

### [To Table of Contents](#TOC)

# The Catalog <a name="catalog"></a>

After connection to the API, create an instance of the **catalog**, which allows you to browse through the following risk model parameters in the QES database:

* <ins>**Universe:**</ins> A time varying collection of stocks. We have wide array of universe available. Additional universes can be added to the collection on client's request. 


* <ins>**Factor:**</ins> List of available factors (or features, e.g., Earnings Yield, ROE, ..) for stocks. The factors form the basis for the risk model and capturing the systematic risk. Factors can be added to the risk model template to enhance the model. While adding more factor can better capture the systematic risk, caution should be exercised, as too many factors could lead to overfitting. Additionally, it is advisable to add correlated factors as they can cause multi-collinearity. 


* <ins>**Meta Factor:**</ins> List of meta data available (e.g., Market Cap, Ticker, ..) for stocks. This is for reference and mapping purposes. 


* <ins>**Portfolio:**</ins> User can upload a portfolio to the database. Portfolios are saved in the users own space. The purpose of the portfolio is to be used as a custom universe for the risk model and other requests. 


* <ins>**Template:**</ins> A template is stored parameter set for the risk model requests. The two main purposes of templates are 
    1) Simplifying the API, and 
    2) Reusing parameters across different requests. 

In [59]:
# creting instance of the catalog
catalog = connection.get_catalog()

### <u>Universes<a name="univ"></a> 

Below are 6 of the commonly used universes, but many more are offered (call to head() function can be removed to see the rest). Note that custom portfolios can also be used as risk universe. The ID column in the table below shows the unique identifier used to set the universe. 

In [72]:
catalog.get_universe().head(6)

,ID,NAME,SECTOR,COUNTRY,DESCRIPTION
0,US_1,US Broad Market Index,All,US,Universe stocks from US based on Market Capita...
1,SP500,QES 500,All,US,Universe provides constituents for S&P 500
2,SP1500,QES 1500,All,US,Universe provides constituents for S&P 1500
3,CIQ_INDEX_2667223,Russell 2000 Index,All,US,Russell 2000 Index
4,CIQ_INDEX_2668794,Russell 1000 Index,All,US,Russell 1000 Index
5,CIQ_INDEX_2668795,Russell 3000 Index,All,US,Russell 3000 Index


### <u>Factors<a name="fact"></a> 

We have a wide array of factors available capturing fundamental and technical themes of the stocks. The factors are based on years of extensive alpha/risk research by the QES team. The catolog provides an easy way to list down available factors, and below are some examples of currently available factors. In case you are interested in additional factors, please [email](mailto:luo.qes@wolferesearch.com) desk. 

In [74]:
catalog.get_factors().head()

,CATEGORY,SUBCATEGORY,ID,DESCRIPTION
0,Value,Book,BOOKP_FY1,"Book-to-market, FY1"
1,Value,Earnings,PE_LTM_D,"Price-to-EPS, LTM, diluted"
2,Value,Cash flow,CFOYLD_FY1,"Operating cash flow yield, FY1 (= Last 12 mont..."
3,Value,GARP,EPSYLD_GRO,"Earnings yield (LTM, basic) x 5Y Exp Growth"
4,Value,Earnings,EPSYLD_LTM_O,"Earnings yield, LTM, operating"


### <u>Portfolios<a name="port"></a> 

Similar to a universe but user can upload it along with custom factors. A single portfolio added to the database can provide a risk universe as well as custom factors. Note that any columns other than date/id are treated as custom factors). For more details on format of the CSV file, please see our [Github link](https://github.com/wolferesearch/docs/blob/master/portfolio.md). You can also download a sample file using this [link](https://raw.githubusercontent.com/wolferesearch/docs/master/sample/LongFormatPort.csv). 

In [62]:
catalog.get_portfolios().head()

,ID,UPLOADEDBY,UPLOADEDTIME
0,HJAIN_MY_CUSTOM_PORTFOLIO,hjain,2023-02-01T00:00:00.000Z
1,Custom_Port,hjain,2019-02-13T00:00:00.000Z
2,A1A2,hjain,2019-02-13T00:00:00.000Z
3,hjPortDemo,hjain,2019-01-29T00:00:00.000Z
4,TrialPort,hjain,2019-01-29T00:00:00.000Z


### <u>Templates<a name="temp"></a> 

Templates are stored parameters that can be reused. Templates are stored as Json files in the database. Users can take an existing template, modify it, and save it back in the database. The template will be stored under the individual users space and only visible to the account you are using to connect. 

In [64]:
catalog.get_templates().head()

,CONTENT,OWNER,NAME,DESCRIPTION,MODIFIED_DATE,TYPE
0,"{'__type__': 'ltool.risk.RiskModelBuilder', '_...",admin,default,Model uses default set of factors and grouping...,2019-01-18T16:34:39.000000,Risk-Model
1,"{'localMode': 1, '__type__': 'ltool.risk.RiskM...",admin,default-euro,Model is build for Euro with default currency ...,2023-01-04T15:42:10.000000,Risk-Model
2,"{'__type__': 'ltool.opt.RiskModelBuilder', '__...",hjain,default3,default3,2019-01-29T23:28:00.000000,Risk-Model
3,"{'__type__': 'ltool.risk.RiskModelBuilder', '_...",hjain,default-hj1,default-hj1,2019-01-17T22:51:04.000000,Risk-Model
4,"{'__type__': 'ltool.opt.RiskModelBuilder', '__...",hjain,default37,default37,2019-01-29T23:28:19.000000,Risk-Model


<span style="color:red">*Note: The universe, factor, and template IDs, and the Template NAME are used as string parameters when setting the universe or adding factors to the model </span>

### [To Table of Contents](#TOC)

# Building a Custom "On the Fly" Risk Model <a name="riskmodel"></a>

For this guide, we are going to be using the default risk model template as an example. However, any of the available templates can be used as a base for a custom model.

Use the values from the NAME and TYPE columns of the templates data frame as the parameters to set the template of your model.

In [67]:
example_risk_template = connection.get_template(name = 'default', type_ = 'Risk-Model')

##### To see the factors and meta data used in the template, print the json form of the Risk Model Template object.

In [68]:
example_risk_template.json

{'__type__': 'ltool.risk.RiskModelBuilder',
 '__name__': 'default',
 'factors': [{'name': 'Earnings Yld', 'mnemonic': 'EPSYLD_LTM_B'},
  {'name': 'EPS Growth (YoY)', 'mnemonic': 'GR_INTR_EPS'},
  {'name': 'Momentum (12M-1M)', 'mnemonic': 'RTN_12M1M'},
  {'name': 'Revision', 'mnemonic': 'ES_EPS_NTM_R3M'},
  {'name': 'Profitability', 'mnemonic': 'ROE'},
  {'name': 'Volatility', 'mnemonic': 'REAL_VOL'},
  {'name': 'Size (Mkt. Cap)', 'mnemonic': 'MKTCAP_M_USD'},
  {'name': 'Div Yield', 'mnemonic': 'DIVYLD_TRL'},
  {'name': 'Book to Market', 'mnemonic': 'BOOKP'}],
 'options': {'spRisk': {'fn': 'median', 'shrinkage': 0.5}},
 'meta': [{'name': 'Sedol', 'mnemonic': 'SEDOL'},
  {'name': 'Ticker', 'mnemonic': 'TICKER'},
  {'name': 'Company Name', 'mnemonic': 'COMPANYNAME'},
  {'name': 'Sector', 'mnemonic': 'QES_GSECTOR'},
  {'name': 'Industry Group', 'mnemonic': 'QES_GGROUP'},
  {'name': 'Country', 'mnemonic': 'QES_COUNTRY'},
  {'name': 'Currency', 'mnemonic': 'CURRENCY'}],
 'covArgs': {'interva

##### Adding a new factor to the template:

In [75]:
example_risk_template.add_factor(mnemonic = '-RTN21D', name = 'One Month Reversion')

##### Saving template to the database:

In [76]:
# use the following convention to name and save the template
template_name = '{}_risk_with_1m_reversion'.format(user_name)
example_risk_template.save(template_name)

##### Ensuring the new template was saved to the catalog:

In [244]:
catalog.get_templates()

,CONTENT,OWNER,NAME,DESCRIPTION,MODIFIED_DATE,TYPE
0,"{'__type__': 'ltool.risk.RiskModelBuilder', '_...",admin,default,Model uses default set of factors and grouping...,2019-01-18T16:34:39.000000,Risk-Model
1,"{'localMode': 1, '__type__': 'ltool.risk.RiskM...",admin,default-euro,Model is build for Euro with default currency ...,2023-01-04T15:42:10.000000,Risk-Model
2,"{'__type__': 'ltool.opt.RiskModelBuilder', '__...",hjain,default3,default3,2019-01-29T23:28:00.000000,Risk-Model
3,"{'__type__': 'ltool.risk.RiskModelBuilder', '_...",hjain,default-hj1,default-hj1,2019-01-17T22:51:04.000000,Risk-Model
4,"{'__type__': 'ltool.opt.RiskModelBuilder', '__...",hjain,default37,default37,2019-01-29T23:28:19.000000,Risk-Model
...,...,...,...,...,...,...
18,"{'__type__': 'ltool.risk.RiskModelBuilder', 'n...",hjain,hjain_risk_with_cash_flow,hjain_risk_with_cash_flow,2023-02-01T22:49:25.000000,Risk-Model
19,"{'implied_alpha': False, '__type__': 'ltool.op...",admin,default-empty,Model uses default optimization,2023-05-17T19:07:38.000000,Optimization
20,"{'soft_ADV_trading_penalty': 1000, '__type__':...",admin,default-adv,Model uses default optimization,2023-05-24T17:29:04.000000,Optimization
21,"{'min_weight': 0.001, 'max_neutral_exposure': ...",admin,default,Default Hedging Template,2023-06-27T19:23:33.000000,Portfolio Factor Hedging


##### Ensuring the new template has the new factor:

In [77]:
## Get Template
risk_template = connection.get_template(name = template_name, type_ = 'Risk-Model')
risk_template.json

{'__type__': 'ltool.risk.RiskModelBuilder',
 'name': 'hjain_risk_with_1m_reversion',
 '__name__': 'hjain_risk_with_1m_reversion',
 'factors': [{'name': 'Earnings Yld', 'mnemonic': 'EPSYLD_LTM_B'},
  {'name': 'EPS Growth (YoY)', 'mnemonic': 'GR_INTR_EPS'},
  {'name': 'Momentum (12M-1M)', 'mnemonic': 'RTN_12M1M'},
  {'name': 'Revision', 'mnemonic': 'ES_EPS_NTM_R3M'},
  {'name': 'Profitability', 'mnemonic': 'ROE'},
  {'name': 'Volatility', 'mnemonic': 'REAL_VOL'},
  {'name': 'Size (Mkt. Cap)', 'mnemonic': 'MKTCAP_M_USD'},
  {'name': 'Div Yield', 'mnemonic': 'DIVYLD_TRL'},
  {'name': 'Book to Market', 'mnemonic': 'BOOKP'},
  {'name': 'One Month Reversion', 'mnemonic': '-RTN21D'}],
 'options': {'spRisk': {'fn': 'median', 'shrinkage': 0.5}},
 'meta': [{'name': 'Sedol', 'mnemonic': 'SEDOL'},
  {'name': 'Ticker', 'mnemonic': 'TICKER'},
  {'name': 'Company Name', 'mnemonic': 'COMPANYNAME'},
  {'name': 'Sector', 'mnemonic': 'QES_GSECTOR'},
  {'name': 'Industry Group', 'mnemonic': 'QES_GGROUP'},


## <u>Risk Model Builder Class<a name="rmb"></a>

The Risk Model Builder requires the following parameters to build the risk model:

1. <u>Universe:</u> This can be one of the standard universes or custom portfolio ids
2. <u>Template:</u> Template containing the parameters such as factor lists, meta data list, and other parameters
3. <u>Start Date:</u> First date when the risk model should be built
4. <u>End Date:</u> Last date when the risk model should be built
5. <u>Frequency:</u> How often to rebuild between the start and end date. Available frequencies are 
   1. Daily:  1d
   2. N Daily: nd
   3. Weekly: 1w
   4. Monthly: 1m
   5. Month End: 1me
   6. Quarterly: 1q
   7. Quarter End: 1qe
   8. Yearly: 1y
   9. Year End: 1ye

In [163]:
# Get a new Risk model builder
rmb = connection.get_risk_model_builder()

# Making a new risk model using the _ investment universe at monthly frequency
rmb.new_request(universe = 'CIQ_INDEX_2668795', template = template_name, startDate = '2020-01-01', endDate = '2022-12-31', freq = '1m')

##### new_request() function from the RiskModel class automatically submits job request (job requests covered in next section).

In [322]:
# allow the risk model time to build before displaying the info - will be explained in next section
rmb.wait(max_wait_secs = 600)
rmb.info()

{'type': 1,
 'status': 'SUCCESS',
 'message': 'Job Completed ',
 'uuid': '62df86d3-a931-422b-b4d9-c9db4fe35a83',
 'endTime': '"2023-06-22 15:03:05.263436"',
 'startTime': '"2023-06-22 15:00:03.912968"',
 'dates': ['2020-01-01',
  '2020-02-01',
  '2020-03-01',
  '2020-04-01',
  '2020-05-01',
  '2020-06-01',
  '2020-07-01',
  '2020-08-01',
  '2020-09-01',
  '2020-10-01',
  '2020-11-01',
  '2020-12-01',
  '2021-01-01',
  '2021-02-01',
  '2021-03-01',
  '2021-04-01',
  '2021-05-01',
  '2021-06-01',
  '2021-07-01',
  '2021-08-01',
  '2021-09-01',
  '2021-10-01',
  '2021-11-01',
  '2021-12-01',
  '2022-01-01',
  '2022-02-01',
  '2022-03-01',
  '2022-04-01',
  '2022-05-01',
  '2022-06-01',
  '2022-07-01',
  '2022-08-01',
  '2022-09-01',
  '2022-10-01',
  '2022-11-01',
  '2022-12-01']}

<span style="color:red">*Note: Give each job request time to complete using the wait() function before proceeding or checking info

## <u>Getting the Data from the Request<a name="req_data"></a>

Risk model data can be accessed by querying it one date at a time. The data is returned as a dictionary containing the following data for one day:

1. [Factor Covariance Matric (*.cov)](#.cov)
1. [Security Exposures (*.exp)](#.exp)
1. [Factor Mimicking Portfolio (*.fmp)](#.fmp)
1. [Security Master (*.idm)](#.idm)
1. [Factor Info file (*.info)](#.info)
1. [Security Meta (*.meta)](#.meta)
1. [Security Specific Risk (*.rsk)](#.rsk)
1. [Factor Returns (over the period) (*.rtn)](#.rtn)



<span style="color:red">*Note: The data of each risk model request is kept in the database based on unique ID (UID) of the request. The UID can be retrieved using the connection.jobs() function of a previous request. You can always query an old request by associating the builder with previous run UID</span>

##### Say we want to see the risk model data from December 31, 2022:

In [165]:
data = rmb.get_data('2022-12-01')

##### Print the names of each matrix for reference:

In [252]:
for k in data.keys():
    print(k + "\n")

2022-12-01/R1_D1_20221201.cov

2022-12-01/R1_D1_20221201.exp

2022-12-01/R1_D1_20221201.fmp

2022-12-01/R1_D1_20221201.idm

2022-12-01/R1_D1_20221201.info

2022-12-01/R1_D1_20221201.meta

2022-12-01/R1_D1_20221201.rsk

2022-12-01/R1_D1_20221201.rtn



### <u>Factor Covariance Matrix (*.cov)<a name=".cov"></a>

Factor covariance matrix provides ex-ante covariance based on the risk model estimation as of that point in time. The risk of a security or a portfolio can be estimated by combining the exposure with the covariance. The covariance values are annualized. The diagonal element is the estimation of volatility of the corresponding factor. 

In [167]:
data['2022-12-01/R1_D1_20221201.cov'].head()

,Earnings Yld,EPS Growth (YoY),Momentum (12M-1M),Revision,Profitability,Volatility,Size (Mkt. Cap),Div Yield,Book to Market,One Month Reversion,...,Banks,Diversified Financials,Insurance,Software & Services,Technology Hardware & Equipment,Semiconductors & Semiconductor Equipment,Telecommunication Services,Media & Entertainment,Utilities,Real Estate
Earnings Yld,0.000741,-0.000045,0.000457,-0.000083,-0.000164,0.000691,-0.000031,0.000011,0.000202,-0.000199,...,0.003012,0.002347,0.002107,0.001609,0.002217,0.002172,0.001212,0.001901,0.000768,0.001983
EPS Growth (YoY),-0.000045,0.000129,-0.000075,0.000027,-0.000101,0.000415,0.000096,-0.000069,-0.000021,0.000036,...,0.001391,0.001514,0.001375,0.001376,0.001177,0.001360,0.000871,0.001287,0.001199,0.001561
Momentum (12M-1M),0.000457,-0.000075,0.002798,-0.000174,0.000170,-0.000379,-0.000717,0.000380,0.000957,-0.000590,...,0.000970,-0.000583,-0.000275,-0.003296,-0.000906,-0.001709,-0.000853,-0.000691,-0.002199,0.000318
Revision,-0.000083,0.000027,-0.000174,0.000204,-0.000094,0.000336,0.000002,-0.000048,-0.000094,0.000007,...,0.000811,0.000902,0.000761,0.000793,0.000461,0.000646,0.000516,0.000618,0.000693,0.000825
Profitability,-0.000164,-0.000101,0.000170,-0.000094,0.000651,-0.001282,-0.000069,0.000143,0.000255,-0.000076,...,-0.003016,-0.002951,-0.002548,-0.003223,-0.002640,-0.002982,-0.002096,-0.003177,-0.002233,-0.002763


### <u>Factor Exposure Data (*.exp)<a name=".exp"></a>

The file provides the factor exposure for each of the securities in the risk universe. 

In [168]:
data['2022-12-01/R1_D1_20221201.exp'].head()

,Earnings Yld,EPS Growth (YoY),Momentum (12M-1M),Revision,Profitability,Volatility,Size (Mkt. Cap),Div Yield,Book to Market,One Month Reversion,...,Banks,Diversified Financials,Insurance,Software & Services,Technology Hardware & Equipment,Semiconductors & Semiconductor Equipment,Telecommunication Services,Media & Entertainment,Utilities,Real Estate
ID,,,,,,,,,,,,,,,,,,,,,
04M5VVVP14,0.392177,4.536413,1.994277,0.656668,-2.221902,-0.028021,-0.338015,-2.353098,3.716213,-0.167689,...,0,0,0,0,0,0,0,0,0,0
K9RQE6VQGZ,-0.700313,-0.223514,1.523220,4.742418,-2.330901,1.254442,1.751569,2.959053,2.972070,0.690644,...,0,0,0,0,0,0,0,0,0,0
04M5J6K6G4,-0.057921,2.011848,0.525505,0.926650,-1.204208,-1.779585,1.074743,0.777111,1.611282,-1.038575,...,0,0,0,0,0,0,0,0,1,0
04M5JOJP14,2.873573,3.370150,-1.239433,2.828580,4.484874,0.815644,0.197213,2.922627,0.466594,-0.886606,...,0,1,0,0,0,0,0,0,0,0
04M5JN8NG4,-0.215902,-2.311442,-0.090440,-0.435091,0.982326,-1.438174,2.753373,0.263808,-1.374962,-0.421545,...,0,0,0,0,0,0,0,0,0,0


### <u>Factor Mimicking Portfolio (*.fmp)<a name=".fmp"></a>

Provides composition of the FMP as constructed for that date. 

In [173]:
data['2022-12-01/R1_D1_20221201.fmp'].head()

,Earnings Yld,EPS Growth (YoY),Momentum (12M-1M),Revision,Profitability,Volatility,Size (Mkt. Cap),Div Yield,Book to Market,One Month Reversion,...,Banks,Diversified Financials,Insurance,Software & Services,Technology Hardware & Equipment,Semiconductors & Semiconductor Equipment,Telecommunication Services,Media & Entertainment,Utilities,Real Estate
ID,,,,,,,,,,,,,,,,,,,,,
04M5VVVP14,0.000178,0.000202,0.001160,0.000007,-0.000381,-0.000017,-0.000111,-0.000826,0.000525,-0.000255,...,-0.000425,0.000685,-0.000367,0.000505,-0.000199,0.000703,-0.000039,0.000278,0.000001,0.000606
K9RQE6VQGZ,0.002101,0.000738,-0.000564,0.000667,-0.003904,-0.000376,0.000250,0.000120,-0.002619,0.000755,...,0.000260,0.001388,0.000533,-0.001649,0.000552,0.001032,0.000837,0.000504,-0.000560,0.000391
04M5J6K6G4,0.000310,-0.000499,0.000789,0.000376,-0.000146,0.000430,0.000110,0.000450,0.000696,-0.001159,...,-0.000708,-0.001024,-0.001017,0.000610,-0.000028,-0.000253,-0.000173,0.000259,0.015966,-0.000415
04M5JOJP14,0.000638,-0.000169,-0.000564,-0.000270,-0.000116,0.000105,-0.000186,-0.000891,-0.000285,-0.000349,...,0.000846,0.005214,0.000976,-0.000214,-0.000085,-0.001027,0.000677,-0.000399,0.001303,0.001186
04M5JN8NG4,0.000236,-0.000157,-0.001193,0.000315,0.000200,-0.001095,0.001178,0.000290,0.000172,0.000208,...,-0.001176,-0.001408,-0.001221,-0.000723,-0.000391,-0.000985,-0.000760,-0.000870,-0.002143,-0.001499


### <u>Security Master (*.idm)<a name=".idm"></a>

Basic security master file for each date listing reference information about the company.

In [ ]:
data['2022-12-01/R1_D1_20221201.idm'].head()

,SEDOL,TICKER,COMPANYNAME,QES_GSECTOR,QES_GGROUP,QES_COUNTRY,CURRENCY,IN_RISK_UNIV
ID,,,,,,,,
04M5VVVP14,2001119,AIR,AAR Corp,Industrials,Capital Goods,US,USD,True
K9RQE6VQGZ,BCV7KT2,AAL,American Airlines Group Inc,Industrials,Transportation,US,USD,True
04M5J6K6G4,2048804,PNW,Pinnacle West Capital Corp,Utilities,Utilities,US,USD,True
04M5JOJP14,BLFGN66,PRG,PROG Holdings Inc,Financials,Diversified Financials,US,USD,True
04M5JN8NG4,2002305,ABT,Abbott Laboratories,Health Care,Health Care Equipment & Services,US,USD,True


### <u>Factor Info File (*.info)<a name=".info"></a>

Provides details of each of the factors used in the risk model - includes the sector/country/currency when available. 

In [172]:
data['2022-12-01/R1_D1_20221201.info'].head()

,Category,FactorAlias
FactorName,,
EPSYLD_LTM_B,Style,Earnings Yld
GR_INTR_EPS,Style,EPS Growth (YoY)
RTN_12M1M,Style,Momentum (12M-1M)
ES_EPS_NTM_R3M,Style,Revision
ROE,Style,Profitability


### <u>Security Meta (*.meta)<a name=".meta"></a>
Gives insight into each individual security's performance and contribution to the overall returns of the portfolio.

In [228]:
data['2022-12-01/R1_D1_20221201.meta'].head()

,SEDOL,TICKER,COMPANYNAME,QES_GSECTOR,QES_GGROUP,QES_COUNTRY,CURRENCY,IN_RISK_UNIV,TotalRisk,SpecificRisk,...,Banks,Diversified Financials,Insurance,Software & Services,Technology Hardware & Equipment,Semiconductors & Semiconductor Equipment,Telecommunication Services,Media & Entertainment,Utilities,Real Estate
SecurityID,,,,,,,,,,,,,,,,,,,,,
04M5VVVP14,2001119,AIR,AAR Corp,Industrials,Capital Goods,US,USD,True,52.063848,29.066798,...,0,0,0,0,0,0,0,0,0,0
K9RQE6VQGZ,BCV7KT2,AAL,American Airlines Group Inc,Industrials,Transportation,US,USD,True,61.731742,38.707941,...,0,0,0,0,0,0,0,0,0,0
04M5J6K6G4,2048804,PNW,Pinnacle West Capital Corp,Utilities,Utilities,US,USD,True,26.743368,16.112068,...,0,0,0,0,0,0,0,0,1,0
04M5JOJP14,BLFGN66,PRG,PROG Holdings Inc,Financials,Diversified Financials,US,USD,True,54.574007,36.928720,...,0,1,0,0,0,0,0,0,0,0
04M5JN8NG4,2002305,ABT,Abbott Laboratories,Health Care,Health Care Equipment & Services,US,USD,True,36.630546,30.016360,...,0,0,0,0,0,0,0,0,0,0


### <u>Security Idio Risk (*.rsk)<a name=".rsk"></a>

The idio risk provides a security specific risk. The units are percentage, so the values should be divided by 100 prior to adding to the risks of securities

In [169]:
data['2022-12-01/R1_D1_20221201.rsk'].head()

,Specific Risk
ID,
04M5VVVP14,0.290668
K9RQE6VQGZ,0.387079
04M5J6K6G4,0.161121
04M5JOJP14,0.369287
04M5JN8NG4,0.300164


### <u>Factor Returns File (*.rtn)<a name=".rtn"></a>

The factor returns provide the percentage return. The returns are computed over the queried frequency. In order to compute decimal returns, the numbers should be divided by 100. 

In [170]:
data['2022-12-01/R1_D1_20221201.rtn'].head()

,Return
FactorName,
Earnings Yld,-0.124477
EPS Growth (YoY),-0.352558
Momentum (12M-1M),-1.459960
Revision,-0.605994
Profitability,0.622202


### [To Table of Contents](#TOC)

# Job Requests<a name="job_reqs"></a>

The Risk Model/Risk Model Builder, Optimizer, and Attribution classes are all job runner classes and sub-classes of the Base class. The Base class holds functions related to getting information related to job requests. For example, checking the status of the job or getting the job logs.

<span style="color:red">*Note: All commands to follow are available to all job runner classes. They will be demonstrated with the instance of the RiskModelBuilder Class made above, but 'rmb' can be switched with the variable name of the job runner class of the desired job.</span>

## <u>Job Request Content<a name="req_cont"></a>
Each instance of a job request class contains a job request object. Before submitting a job request, print out the job request object content to ensure all desired parameters are set.

In [265]:
rmb.req

{'universe': 'CIQ_INDEX_2668795',
 'template': 'hjain_risk_with_1m_reversion',
 'startDate': '2020-01-01',
 'endDate': '2022-12-31',
 'freq': '1m'}

#### Once the request object holds the desired parameters, the job can be submitted with the <u>submit()</u> function.

# <u>Job Info<a name="job_info"></a>
After a job is submitted, the info() function can be run to see the returned value from the server. Before checking the job info, allow the job time to complete using the wait() function. If info() is run before the job is completed, it will contain Error messages and the status will always be either STARTED or ERROR.

In [282]:
rmb.wait(max_wait_secs=600)
rmb.info()

{'type': 1,
 'status': 'SUCCESS',
 'message': 'Job Completed ',
 'uuid': '62df86d3-a931-422b-b4d9-c9db4fe35a83',
 'endTime': '"2023-06-22 15:03:05.263436"',
 'startTime': '"2023-06-22 15:00:03.912968"',
 'dates': ['2020-01-01',
  '2020-02-01',
  '2020-03-01',
  '2020-04-01',
  '2020-05-01',
  '2020-06-01',
  '2020-07-01',
  '2020-08-01',
  '2020-09-01',
  '2020-10-01',
  '2020-11-01',
  '2020-12-01',
  '2021-01-01',
  '2021-02-01',
  '2021-03-01',
  '2021-04-01',
  '2021-05-01',
  '2021-06-01',
  '2021-07-01',
  '2021-08-01',
  '2021-09-01',
  '2021-10-01',
  '2021-11-01',
  '2021-12-01',
  '2022-01-01',
  '2022-02-01',
  '2022-03-01',
  '2022-04-01',
  '2022-05-01',
  '2022-06-01',
  '2022-07-01',
  '2022-08-01',
  '2022-09-01',
  '2022-10-01',
  '2022-11-01',
  '2022-12-01']}

## <u>Job Logs<a name="job_logs"></a>
This will show the response from the output from the server while the job was running. Mainly useful for checking Error messages if job fails.

In [264]:
print(rmb.get_logs())

2023-06-22 15:01:23 - INFO - Running for 2022-09-01
2023-06-22 15:01:23 - INFO - Building Covariance Matrix for ==> 2022-09-01
2023-06-22 15:01:32 - INFO - Building Covariance Matrix for ==> 2021-12-01
[1] "Length of the filtered stocks"
[1] 3028
2023-06-22 15:01:42 - INFO - esti name: 3028
 no esti names: 0
2023-06-22 15:01:42 - INFO - esti ind: 24
 no esti ind: 0
[1] "Length of the filtered stocks"
[1] 3065
2023-06-22 15:01:44 - INFO - esti name: 3065
 no esti names: 0
2023-06-22 15:01:44 - INFO - esti ind: 24
 no esti ind: 0
[1] "QES_GSECTOR"
[1] "QES_GGROUP"
[1] "CURRENCY"
2023-06-22 15:01:50 - INFO - Completed for 2022-01-01
2023-06-22 15:01:50 - INFO - Running for 2022-02-01
2023-06-22 15:01:50 - INFO - Building Covariance Matrix for ==> 2022-02-01
2023-06-22 15:01:59 - INFO - Building Covariance Matrix for ==> 2022-01-01
[1] "Length of the filtered stocks"
[1] 3065
2023-06-22 15:02:07 - INFO - esti name: 3065
 no esti names: 0
2023-06-22 15:02:07 - INFO - esti ind: 24
 no esti i

## <u>Searching For a Job<a name="find_job"></a>
This is not always needed, but can be useful in ensuring a job request has been completed.

Can get the Unique ID of the job from the output of the info() function, or by using the command below for a quicker run time.

In [300]:
# finding unique idea for job request to make a new risk model
rmb.esvc.get_id

'62df86d3-a931-422b-b4d9-c9db4fe35a83'

##### Query the quantDB for the list of jobs:

In [209]:
from lquantPy import LQuant

In [210]:
wq=LQuant.LQuant()
jobs_list = wq.quantdb_query(name='quantDB', query='select * from micsvc_owner.job_queue')

2023-06-27 14:53:10,461 - lquantPy.LQuant - INFO - Initial LQuant. This may take some time...
2023-06-27 14:53:10,464 - lquantPy.LQuant - INFO - Initialized LQuant environment
2023-06-27 14:53:10,466 - lquantPy.LQuant - INFO - Initializing LQuant, This will take some time....


Library Path -Djava.library.path=/usr/local/lib/R/site-library/rJava/jri
Licensed User: 
	 Version: 1.0
	 Module: lquant
	 User: Wolfe Research
	 Company: Wolfe Research
	 Date of Issue :2022-09-28
	 License Valid Until:2024-12-31
	 Days to Expiry: 553
	 Is Valid : true
FileBasedStreamSource [dir=/mnt/ebs7/data/cache, expiry=86400000]
FileBasedStreamSource [dir=/mnt/ebs1/data/cache, expiry=86400000]
FileBasedStreamSource [dir=/mnt/ebs7/data/cache, expiry=86400000]
FileBasedStreamSource [dir=/mnt/ebs1/data/cache, expiry=86400000]
Application Info:
 Number of Data Sources(Connections) : 7
 Number of Attribute Sources         : 385
 Number of Universe Sources          : 89
 Number of Attributes(Factors)       : 176657
 Number of Universes                 : 51563
 Connections:[quantDB, qdb3, quantDB3, quantDB2, qdb2, mysqltest, Quant DB Read]


##### Use 'loc' to search the data frame for the specific job based on the UUID:

In [222]:
jobs_list.loc[jobs_list['UUID'] == '62df86d3-a931-422b-b4d9-c9db4fe35a83']

,TYPEID,USER_,UUID,STARTTIME,STATUS,ENDTIME,MESSAGE
8375,1,hjain,62df86d3-a931-422b-b4d9-c9db4fe35a83,2023-06-22T00:00:00.000Z,SUCCESS,2023-06-22T00:00:00.000Z,Job Completed


### [To Table of Contents](#TOC)

# Running an Attribution on Origional Portfolio <a name="attr1"></a>
This is to see the sources or risk and returns, as well as exposure to different risks and factors.

For this guide, we will be using a sample portflio, who's data is in <u>**PortSimulator.csv**</u>. This file contains 547 different stocks with weights across 24 trading days for each one, with the following order of information for the columns:
* <u>Column 1:</u> The index in the data set
* <u>Column 2:</u> The date of the trading day
* <u>Column 3:</u> The initial stock weights of the portfolio
* <u>Column 4:</u> The stock ticker symbol

In [179]:
portfolio_df = pd.read_csv('PortSimulator.csv')
portfolio_df.head()

,DATE,WEIGHT,TICKER
0,2021-01-29,0.000325,AAL
1,2021-01-29,0.000258,PNW
2,2021-01-29,0.006676,ABT
3,2021-01-29,0.003139,AMD
4,2021-01-29,0.001797,APD


In [329]:
portfolio_df.DATE.unique()

array(['2021-01-29', '2021-02-26', '2021-03-31', '2021-04-30',
       '2021-05-28', '2021-06-30', '2021-07-30', '2021-08-31',
       '2021-09-30', '2021-10-29', '2021-11-30', '2021-12-31',
       '2022-01-31', '2022-02-28', '2022-03-31', '2022-04-29',
       '2022-05-31', '2022-06-30', '2022-07-29', '2022-08-31',
       '2022-09-30', '2022-10-31', '2022-11-30', '2022-12-30'],
      dtype=object)

In [174]:
# Use connection object to get a new instance of the attribution class
attribution = connection.get_attribution()

#### Setting Risk Model
The User can run at attribution using one of the models available stored in a seperate database. The 'Model' column has the String IDs that correspont to the available models.

| Model        | Description |
| -----------  | ----------- |
| QES_US_AC_2  | US Broad Risk Model        |
| QES_US_CON_2  | US Consumers Risk Model        |
| QES_US_ENG_2  | US Energy Risk Model        |
| QES_US_FIN_2  | US Financial Risk Model        |
| QES_US_HC_2  | US Health Care Risk Model        |
| QES_US_IND_2  | US Industrial Risk Model        |
| QES_US_TMT_2  | US TMT (Tech/Media/Telecom) Risk Model        |
| QES_EU_AC_2  | US TMT (Tech/Media/Telecom) Risk Model        |



In [190]:
_ = attribution.set_risk_model('QES_US_AC_2')

##### Upldoading portfolio dataframe to run attribution on:

In [191]:
_ = attribution.set_user_data(name = 'attribution_data.csv', data = portfolio_df)

Calling Uploading /tmp/tmpgjm6xak2.csv to attribution_data.csv ...
Portfolio Uploaded successfully


##### Setting the weight factor for the portfolio as the WEIGHT column in the data frame:

In [192]:
_ = attribution.set_weight_factor(weight_factor = 'WEIGHT')

##### Submitting the job request:

In [289]:
attribution.req

{'user_data': {'format': 'csv', 'name': 'attribution_data.csv'},
 'weightAttribute': 'WEIGHT',
 'risk_model': 'QES_US_AC_2'}

In [290]:
attribution.submit()

In [291]:
attribution.wait(max_wait_secs = 600)
attribution.info()

{'type': 3,
 'status': 'SUCCESS',
 'message': 'Job Completed 0',
 'uuid': '723703a7-941d-483f-b0d6-d2b1ff07aba9',
 'endTime': '"2023-06-28 16:14:00.514460"',
 'startTime': '"2023-06-28 16:12:48.388068"'}

## <u> Getting Results from the Attribution<a name="attr1res"></a> 

In [292]:
att_output = attribution.get_output()

In [293]:
att_output.files

,Key,LastModified,Size
0,MN_summary.csv,2023-06-28 16:14:01+00:00,6401
1,ts_data/MN_CORRELATION.csv,2023-06-28 16:14:01+00:00,21565
2,ts_data/MN_CUM_RETURN_CONTRI.csv,2023-06-28 16:14:01+00:00,19984
3,ts_data/MN_DAILY_CUM_RETURN_CONTRI.csv,2023-06-28 16:14:01+00:00,419252
4,ts_data/MN_DAILY_FACTOR_CUM_RETURN.csv,2023-06-28 16:14:01+00:00,418378
...,...,...,...
12,ts_data/MN_RETURN_CONTRI.csv,2023-06-28 16:14:01+00:00,23852
13,ts_data/MN_RISK_CONTRI.csv,2023-06-28 16:14:01+00:00,23938
14,ts_data/MN_RISK_CONTRI_PCT.csv,2023-06-28 16:14:01+00:00,22905
15,ts_data/MN_VOL_ADJ_EXPOSURE.csv,2023-06-28 16:14:01+00:00,23507


### <u>Correlations<a name="corr1"></a>

In [294]:
att_output.get_data('ts_data/MN_CORRELATION.csv')

{'ts_data': {'CORRELATION':               EPSYLD     BOOKP    GROWTH  REVISIONS    PROFIT  LEVERAGE  \
  2021-01-29 -0.053394  0.104933 -0.104867   0.061656  0.197603  0.091823   
  2021-02-26 -0.050100  0.087042 -0.023798   0.244586  0.148120  0.092939   
  2021-03-31 -0.083136  0.095663 -0.059624   0.065965  0.082454  0.169904   
  2021-04-30 -0.081307  0.080761  0.026641   0.108778  0.067306  0.166764   
  2021-05-28 -0.012319  0.062910  0.186909   0.116562 -0.043245  0.157552   
  ...              ...       ...       ...        ...       ...       ...   
  2022-08-31 -0.096143 -0.101110  0.104767  -0.050546  0.071882  0.002064   
  2022-09-30 -0.074843 -0.061736  0.084980   0.019429  0.056171  0.074437   
  2022-10-31  0.002308 -0.098839  0.055002   0.042431  0.018172  0.035090   
  2022-11-30 -0.016314 -0.075543  0.076205  -0.065360  0.012626  0.081354   
  2022-12-30 -0.042847 -0.055859  0.025268   0.035968  0.021210  0.069164   
  
                DIVYLD  MOMENTUM  VOLATILITY   

In [295]:
summary = att_output.get_data('MN_summary.csv')

In [296]:
summary['summary']

,EXPOSURE,FACTOR_DECILE,VOL_ADJ_EXPOSURE,CORRELATION,RISK_CONTRI,RISK_CONTRI_PCT,RETURN_CONTRI
EPSYLD,-0.412321,4.125000,-0.011986,-0.072754,0.000967,0.004707,-0.052365
BOOKP,-0.457284,3.833333,-0.012248,0.009938,-0.000295,-0.001458,-0.008348
GROWTH,-0.069930,5.250000,-0.000707,0.161767,-0.000437,-0.002012,-0.010208
REVISIONS,-0.109851,5.000000,-0.001768,0.015911,-0.000047,-0.000198,-0.002402
PROFIT,0.714197,8.125000,0.012528,0.094323,0.001173,0.005591,0.015097
...,...,...,...,...,...,...,...
STYLE,1.000000,NaN,0.093514,-0.185005,-0.017822,-0.083898,0.009523
SECTOR,1.000000,NaN,0.246234,0.920418,0.226733,1.069601,0.015645
SYSTEMATIC,1.000000,NaN,0.210448,0.992582,0.208911,0.985704,0.025168
RESIDUAL,1.000000,NaN,0.025470,0.117324,0.002987,0.014296,0.034956


### <u>Factor Exposures<a name="fact_exp1"></a>

In [297]:
att_result = attribution.get_results()

In [298]:
att_result.get_exposures()

,EPSYLD,BOOKP,GROWTH,REVISIONS,PROFIT,LEVERAGE,DIVYLD,MOMENTUM,VOLATILITY,SIZE,...,SEMISEQUIP,SOFTWARE,TELECOMM,TRANSPORTATION,UTILITIES,STYLE,SECTOR,SYSTEMATIC,RESIDUAL,TOTAL
2021-01-29,-0.343527,-0.480544,0.640339,0.216187,0.974663,-0.135413,-0.041869,0.341468,-0.985695,1.800685,...,0.051423,0.136320,0.018307,0.018094,0.026436,1,1,1,1,1
2021-02-26,-0.217038,-0.483714,0.433713,-0.022795,0.878686,-0.193207,-0.032488,0.132811,-0.958078,1.733308,...,0.053487,0.137387,0.017546,0.019104,0.024219,1,1,1,1,1
2021-03-31,-0.202226,-0.504902,0.411034,-0.030753,0.874965,-0.104434,0.047851,-0.403089,-0.898598,1.692701,...,0.053421,0.133130,0.017845,0.019820,0.025555,1,1,1,1,1
2021-04-30,-0.211662,-0.508340,0.478362,-0.046954,0.975462,-0.126870,0.061044,-0.437416,-0.865519,1.660379,...,0.050974,0.135291,0.017366,0.019772,0.025340,1,1,1,1,1
2021-05-28,-0.213194,-0.532811,-0.048262,-0.036435,0.982275,-0.228712,0.039309,-0.368796,-0.822195,1.698332,...,0.052268,0.133183,0.017115,0.020305,0.024548,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-08-31,-0.501168,-0.360595,-0.326614,-0.283448,0.482583,-0.395035,0.075417,0.294153,-0.580037,1.772050,...,0.049025,0.130971,0.014169,0.017549,0.030239,1,1,1,1,1
2022-09-30,-0.577020,-0.386216,-0.372881,-0.308189,0.441578,-0.353690,0.143196,0.307866,-0.585991,1.774856,...,0.046182,0.128009,0.014114,0.016347,0.029505,1,1,1,1,1
2022-10-31,-0.514615,-0.405036,-0.410170,-0.222140,0.462096,-0.346526,0.214126,0.313741,-0.608839,1.771885,...,0.045273,0.126282,0.014226,0.016151,0.028690,1,1,1,1,1
2022-11-30,-0.459295,-0.424709,-0.463532,-0.213102,0.497015,-0.377065,0.219263,0.167203,-0.596967,1.786508,...,0.050785,0.127695,0.013893,0.017018,0.029159,1,1,1,1,1


### [To Table of Contents](#TOC)

# Portfolio Optimization <a name="opt1"></a>

#### The Basic mean variance optimization is described as:

Given a portfolio with expected alpha $\alpha$ and covariance matrix $\Sigma$, the mean variance optimization problem seeks to find the weights $w$ that minimize the portfolio risk (measured by the portfolio variance) while achieving a desired level of expected return.

There can be 3 different varition of Portfolio optimization. The generic MVO formulation allows you to use the $\lambda$ parameter to navigate the efficient frontier. 

1. Mean Variance Optimization with Risk Aversion Parameter ($\lambda$): 

    Minimize:  $\lambda \frac{1}{2}w^T\Sigma w - w.\alpha$

    Here w is the weight, $\Sigma$ is the variance-covariance matrix, and $\alpha$ is the expected return. 


2. Minimize Risk:
    $w^T\Sigma w$

3. Maximize Alpha:
    $w.\alpha$


All these objectives can be subjected to constraints (i.e. $\mu_l < w^T\mu < \mu_b$). Constraints can been equality or inequality forms. They are generally represented by the below equation:

$ \sigma_p^2 = \sum_i \sum_j w_i w_j \sigma_i \sigma_j \rho_{ij} $

## <u> Optimizer Class <a name="opt_class"></a> 

In [283]:
# Use connection object to get a new instance of the optimizer
optimizer = connection.get_optimizer()

In [310]:
_ = optimizer.set_risk_model('QES_US_AC_2')

##### Setting the Alpha:

In [311]:
_ = optimizer.set_alpha('WEIGHT')

### <u>Objective<a name="objec"></a>

The objective specifies the utility function to be used by the optimizer to minimize of maximize. Use the following mnemonics for each objective option:

1. 'MVO' - Mean-Variance Optimization. This is the most common optimization. 
1. 'MRO' - Mean-Risk Optimization. Similar to MVO but instead of Variance term in the utility function, we have the risk term.
1. 'maxAlpha' - Maximize Alpha. The optimizer tries to find the solution that maximizes the alpha of the final portfolio. 
1. 'minRisk' - Minimum Risk Portfolio

In [288]:
_ = optimizer.set_objective('MVO')

##### Since we are doing a MVO, we need to set a value for lambda ($\lambda > 0$). This is the risk aversion variable.
* Higher $\lambda$ = more concerned with minimizing risk and willing to sacrifice returns.
* Lower $\lambda$ = taking on more risk for the potential of maximizing returns.

In [312]:
_ = optimizer.set_lambda(0.1)

### <u>Constraints<a name="const"></a>

Both total longs and shor weights can be constrained with total minimum/maximum values. This allows for the optimizer to build different strategies, such as: Long/Short neutral, 130/30, Pure Short, Pure Long, etc.

##### Portfolio constraint setting functions (double input between 0.0 and 1.0 for each one):

In [313]:
_ = optimizer.set_min_long_weight(0.9)
_ = optimizer.set_max_long_weight(0.9)
_ = optimizer.set_min_short_weight(0.1)
_ = optimizer.set_max_short_weight(0.1)

##### Uploading Portfolio:

In [286]:
_ = optimizer.set_user_data(data = portfolio_df, name = 'optimization_data.csv')

Calling Uploading /tmp/tmp0m2tu0w1.csv to optimization_data.csv ...
Portfolio Uploaded successfully


##### Sumbitting the job:

In [314]:
optimizer.req

{'alpha': 'WEIGHT',
 'objective': 'MVO',
 'lambda': 0.1,
 'user_data': {'format': 'csv', 'name': 'optimization_data.csv'},
 'risk_model': {'risk_model_id': 'QES_US_AC_2'},
 'min_long_weight': 0.9,
 'max_long_weight': 0.9,
 'min_short_weight': 0.1,
 'max_short_weight': 0.1}

In [315]:
optimizer.submit()

In [316]:
optimizer.wait(max_wait_secs=600)
optimizer.info()

{'type': 2,
 'status': 'SUCCESS',
 'message': 'Job Completed 0',
 'uuid': 'ce6ad34c-2187-4a8b-9fe2-c304ea6ae5a3',
 'endTime': '"2023-06-29 12:55:49.559696"',
 'startTime': '"2023-06-29 12:51:31.149523"'}

## <u>Inspecting the Result<a name="inspect_res1"></a>

In [317]:
opt_result = optimizer.get_results()

### Get Optimized Weights (Matrix)<a name="opt_weight1"></a>

In [318]:
opt_result.get_weights().head()

,2021-01-29,2021-02-26,2021-03-31,2021-04-30,2021-05-28,2021-06-30,2021-07-30,2021-08-31,2021-09-30,2021-10-29,...,2022-03-31,2022-04-29,2022-05-31,2022-06-30,2022-07-29,2022-08-31,2022-09-30,2022-10-31,2022-11-30,2022-12-30
1045.04,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0
1075.01,0.0,0.0,0.0,0.0,0.000000,0.000001,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000001,0.0
1078.01,0.0,0.0,0.0,0.0,0.000001,0.000001,0.0,0.0,0.0,0.0,...,0.000001,0.000001,0.0,0.000001,0.0,0.0,0.0,0.0,0.000001,0.0
1161.01,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0
1209.01,0.0,0.0,0.0,0.0,0.000000,0.000001,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000001,0.0


### Ex-ante Alpha Based on Optimized Weights<a name="exalpha"></a>

In [319]:
opt_result.get_alpha().head()

,x
2021-01-29,0.026935
2021-02-26,0.025369
2021-03-31,0.024554
2021-04-30,0.025526
2021-05-28,0.024727


### Get Ex-ante Risk of the Portfolio (Timeseries)<a name="exrisk"></a>

In [320]:
opt_result.get_risk().head()

,x
2021-01-29,0.242485
2021-02-26,0.229661
2021-03-31,0.222436
2021-04-30,0.216241
2021-05-28,0.205960


### Get Turnover of the Portfolio (Timeseries)<a name="port_turn"></a>

In [321]:
opt_result.get_turnover().head()

,x
2021-01-29,0.899993
2021-02-26,0.131765
2021-03-31,0.111095
2021-04-30,0.119207
2021-05-28,0.097564


## <u>Creating the Optimized Portfolio</u> (Might not be the right way to do it) <a name="opt_port1"></a>

In [347]:
port_dates = portfolio_df.DATE.unique()
port_dates

array(['2021-01-29', '2021-02-26', '2021-03-31', '2021-04-30',
       '2021-05-28', '2021-06-30', '2021-07-30', '2021-08-31',
       '2021-09-30', '2021-10-29', '2021-11-30', '2021-12-31',
       '2022-01-31', '2022-02-28', '2022-03-31', '2022-04-29',
       '2022-05-31', '2022-06-30', '2022-07-29', '2022-08-31',
       '2022-09-30', '2022-10-31', '2022-11-30', '2022-12-30'],
      dtype=object)

##### The get_portfolio() function returns a pandas data frame with the user data and optimized weights, but only for a single day. To create the portfolio, make a list of the data frames from each day, and then concatinate them.

In [379]:
def make_opt_port(dates, opt_res):
    df_list = []

    for date in dates:
        port_day = opt_res.get_portfolio(date)
        # IMPORTANT: the data frame returned by get_portfolio() does not include the corresponding date, as each data frame in the optimization result is indexed by the date
        port_day['DATE'] = date # need to manually add the date as a row at the end of each "sub data frame"
        df_list.append(port_day)
    
    new_port = pd.concat(df_list)
    
    return new_port

In [376]:
optimized_port = make_opt_port(port_dates, opt_result)

In [377]:
optimized_port.head()

,IN,WEIGHT_x,TICKER,WEIGHT_y,DATE
1045.04,1,0.000325,AAL,0.0,2021-01-29
1075.01,1,0.000258,PNW,0.0,2021-01-29
1078.01,1,0.006676,ABT,0.0,2021-01-29
1161.01,1,0.003139,AMD,0.0,2021-01-29
1209.01,1,0.001797,APD,0.0,2021-01-29


##### Use the command below to save the new optimized portfolio as a .csv file:

In [ ]:
optimized_port.to_csv('file_name.csv')

### [To Table of Contents](#TOC)

## To Do List: <a name = "todo"></a>
IF POSSIBLE: 
* get Consumers, TMT, Healthcare, Industrials, Financials universes from Kartik/Shu for custom model
* find out how to use custom risk model in optimizer/attribution class

To Do:
1. User runs optimization using the initial portfolio, custom risk model(maybe), hedge universe, basket parameters, and QES smart beta hedge framework to generate a tradable hedge basket. [case #1]
1. User receives output including hedge ratio, ex-ante correlation of tradable hedge basket to ideal hedge (is this from optimizer or post-hedge attribution?).
1. User runs attributions on the tradable hedge basket and the initial+hedge portfolio. [case #1]
1. User runs optimizer to construct long-short portfolio that sources idio risk from high conviction stock picks in the initial portfolio and minimizes systematic risks with small weights to other stocks hedge universe.  [case #2]
1. User runs attribution on the optimized portfolio. [case #1]
1. Compare backtested performance and attributions across initial portfolio, initial+hedge portfolio [case #1], and optimized portfolio [case #3]


## Portfolio Simulation (Maybe Include)

### Max Volume Participation Constraint

The backtest can limit the amount it trades on each day. Usually, if the notional is small and there is a good volume on the rebalance day, all shares will be traded on the same day. However, if there is less liquidity on the day, the remaining part trades the next day, so on and so forth. In case there is no liquidity at all, the simulator will throw an error. 



In [ ]:
port_sim = connection.get_portsimulator()

In [ ]:
_ = port_sim.set_max_vol_participation(max_part = 0.06)

### [To Table of Contents](#TOC)